# 00 Run Everything - Publication Workflow

This is the single notebook entry point. Run this first for the complete Coswara pipeline and publication extras. The other notebooks are optional review/debug notebooks only.

Default behavior is conservative: expensive or optional stages are controlled by toggles, and existing outputs are skipped unless `FORCE_REBUILD = True`.

In [13]:
from pathlib import Path
import os
import subprocess
import sys
from datetime import datetime

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "covid_audio_btp").exists():
            return candidate
    raise FileNotFoundError("Could not find project root. Start Jupyter from the extracted covid_audio_btp folder or one of its subfolders.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
print(f"Started: {datetime.now().isoformat(timespec='seconds')}")

Project root: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Python: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python
Started: 2026-06-12T20:21:54


## Configuration

Set only the paths and toggles you need. Keep `RUN_CNN = False` until the classical baseline pipeline is clean.

In [14]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

RAW_COSWARA_DIR = PROJECT_ROOT / "data/raw/coswara"
COUGHVID_RAW = PROJECT_ROOT / "data/raw/coughvid/v3_extracted"

FORCE_REBUILD = True

RUN_ENV_CHECK = True
RUN_LAYOUT_AUDIT = False

RUN_CORE_COSWARA = False
RUN_FEATURES = False
RUN_ML_BASELINES = False
RUN_CALIBRATION = False
RUN_FUSION = False
RUN_CNN = False
RUN_SHIFT_CHECKS = False
RUN_REPORT_ASSETS = False

RUN_PUBLICATION_EXTRAS = True
RUN_METADATA_BASELINE = False
RUN_QUALITY_WEIGHTED_FUSION = False
RUN_ABSTENTION = False
RUN_BOOTSTRAP_CI = False

RUN_COUGHVID_INDEX = False
RUN_COUGHVID_FEATURES = True
RUN_CROSS_DATASET = True
RUN_FEATURE_SHIFT_REPORT = True

RUN_PAPER_TABLES = True
RUN_PAIRED_MODEL_COMPARISON = False
RUN_CONFOUNDING_MATCHING = False
RUN_EXPERIMENT_MANIFEST = True

MIN_COUGH_DETECTED = 0.80
COUGHVID_FEATURE_MAX_ROWS = None

CNN_EPOCHS = 50
CNN_BATCH_SIZE = 8

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Coswara:", RAW_COSWARA_DIR)
print("COUGHVID:", COUGHVID_RAW)
print("Coswara exists:", RAW_COSWARA_DIR.exists())
print("COUGHVID exists:", COUGHVID_RAW.exists())

PROJECT_ROOT: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp
Coswara: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coswara
COUGHVID: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/data/raw/coughvid/v3_extracted
Coswara exists: True
COUGHVID exists: True


## Runner

The helper below runs project scripts, checks required inputs, and skips completed outputs unless forced.

In [15]:
def p(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def path_exists(path):
    return p(path).exists() and p(path).stat().st_size > 0


def missing(paths):
    return [str(path) for path in paths if not path_exists(path)]


def run_step(name, args, enabled=True, requires=None, creates=None, strict_requires=True, force=None):
    requires = requires or []
    creates = creates or []
    force = FORCE_REBUILD if force is None else force
    if not enabled:
        print(f"SKIP {name}: disabled")
        return False
    missing_inputs = missing(requires)
    if missing_inputs:
        message = f"SKIP {name}: missing required input(s): {missing_inputs}"
        if strict_requires:
            raise FileNotFoundError(message)
        print(message)
        return False
    if creates and not force and all(path_exists(path) for path in creates):
        print(f"SKIP {name}: outputs already exist")
        return False
    cmd = [str(item) for item in args]
    print()
    print(f"## {name}")
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return True


CORE_ARTIFACTS = [
    "data/interim/coswara_index.csv",
    "data/processed/metadata_clean.csv",
    "data/interim/modality_availability.csv",
    "data/interim/split_manifest.csv",
    "data/processed/audio_quality.csv",
    "data/processed/features_mfcc.csv",
    "data/processed/spectrogram_index.csv",
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/calibrated_branch_predictions.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]
print("Runner ready")

Runner ready


## Artifact Dashboard

This shows what already exists before the run.

In [16]:
try:
    import pandas as pd
    dashboard = pd.DataFrame(
        [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
    )
    display(dashboard)
except Exception as exc:
    print(f"Dashboard unavailable before dependency install: {exc}")
    for path in CORE_ARTIFACTS:
        print(path, "OK" if path_exists(path) else "missing")

,path,exists,size_bytes
0,data/interim/coswara_index.csv,True,14467114
1,data/processed/metadata_clean.csv,True,15192524
2,data/interim/modality_availability.csv,True,211550
3,data/interim/split_manifest.csv,True,243621
4,data/processed/audio_quality.csv,True,8643031
5,data/processed/features_mfcc.csv,True,90512724
6,data/processed/spectrogram_index.csv,True,3879710
7,data/outputs/metrics/ml_baseline_metrics.csv,True,2362
8,data/outputs/metrics/calibrated_branch_predict...,True,1480378
9,data/outputs/metrics/fusion_metrics.csv,True,538


## Stage 0-1: Environment, Index, Splits, Quality

This stage is mandatory before modeling. It prevents training on bad audio, bad labels, or participant leakage.

In [ ]:
run_step("local preflight", [sys.executable, "scripts/00_local_preflight.py", "--coswara-dir", RAW_COSWARA_DIR], enabled=True, force=True)

run_step("environment check", [sys.executable, "scripts/00_check_environment.py"], enabled=RUN_ENV_CHECK, force=True)

if RUN_CORE_COSWARA and not RAW_COSWARA_DIR.exists():
    raise FileNotFoundError(f"Coswara not found. Put it at {RAW_COSWARA_DIR} or change RAW_COSWARA_DIR.")

run_step(
    "raw layout audit",
    [sys.executable, "scripts/00_inspect_dataset_layout.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "reports/tables/coswara_layout_audit.csv"],
    enabled=RUN_CORE_COSWARA and RUN_LAYOUT_AUDIT,
    creates=["reports/tables/coswara_layout_audit.csv"],
)
run_step(
    "build Coswara index",
    [sys.executable, "scripts/01_build_coswara_index.py", "--raw-dir", RAW_COSWARA_DIR, "--output", "data/interim/coswara_index.csv"],
    enabled=RUN_CORE_COSWARA,
    creates=["data/interim/coswara_index.csv"],
)
run_step(
    "clean metadata",
    [sys.executable, "scripts/02_clean_metadata.py", "--index", "data/interim/coswara_index.csv", "--output", "data/processed/metadata_clean.csv", "--availability-output", "data/interim/modality_availability.csv", "--audit-output", "reports/tables/dataset_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv"],
    creates=["data/processed/metadata_clean.csv", "data/interim/modality_availability.csv"],
)
run_step(
    "participant splits",
    [sys.executable, "scripts/03_create_splits.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/interim/split_manifest.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--audit-output", "reports/tables/split_audit.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/interim/split_manifest.csv"],
)
run_step(
    "quality audit",
    [sys.executable, "scripts/04_quality_audit.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/audio_quality.csv", "--metadata-output", "data/processed/metadata_clean.csv", "--summary-output", "reports/tables/quality_summary.csv"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/split_manifest.csv", "data/processed/metadata_clean.csv"],
    creates=["data/processed/audio_quality.csv"],
)
run_step(
    "artifact validation",
    [sys.executable, "scripts/12_validate_artifacts.py", "--strict"],
    enabled=RUN_CORE_COSWARA,
    requires=["data/interim/coswara_index.csv", "data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"],
    force=True,
)

## Hard Gate

If this cell fails, inspect the optional review notebooks before training models.

In [ ]:
import pandas as pd

from covid_audio_btp.notebook_utils import (
    assert_binary_labels_present,
    assert_no_participant_leakage,
    check_artifacts,
    read_optional_csv,
    stop_if_validation_errors,
)

metadata = pd.read_csv(p("data/processed/metadata_clean.csv"))
quality = pd.read_csv(p("data/processed/audio_quality.csv"))
issues = read_optional_csv("reports/tables/validation_issues.csv")

assert_binary_labels_present(metadata)
assert_no_participant_leakage(metadata)
if issues is not None:
    stop_if_validation_errors(issues)

quality_ok_rate = float((quality["quality_flag"] == "ok").mean()) if len(quality) else 0.0
print(f"quality ok rate: {quality_ok_rate:.3f}")
if quality_ok_rate < 0.50:
    raise AssertionError("Quality ok rate below 50%; review audio quality before modeling.")

## Stage 2-5: Features, Baselines, Calibration, Fusion

This is the main Coswara modeling path. CNN is optional and disabled by default.

In [ ]:
run_step(
    "feature extraction",
    [
        sys.executable,
        "scripts/05_extract_features.py",
        "--metadata",
        "data/processed/metadata_clean.csv",
        "--features-output",
        "data/processed/features_mfcc.csv",
        "--quality-mode",
        "all_samples",
        "--skip-spectrograms",
    ],
    enabled=RUN_FEATURES,
    requires=[
        "data/processed/metadata_clean.csv",
        "data/processed/audio_quality.csv",
    ],
    creates=["data/processed/features_mfcc.csv"],
    force=FORCE_REBUILD,
)
run_step(
    "classical ML baselines",
    [sys.executable, "scripts/06_train_ml_baselines.py", "--features", "data/processed/features_mfcc.csv"],
    enabled=RUN_ML_BASELINES,
    requires=["data/processed/features_mfcc.csv"],
    creates=["data/outputs/metrics/ml_baseline_metrics.csv", "data/outputs/metrics/ml_validation_metrics.csv", "data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
)
run_step(
    "compact CNN cough branch",
    [sys.executable, "scripts/07_train_cnn.py", "--spectrogram-index", "data/processed/spectrogram_index.csv", "--modality", "cough", "--epochs", CNN_EPOCHS, "--batch-size", CNN_BATCH_SIZE],
    enabled=RUN_CNN,
    requires=["data/processed/spectrogram_index.csv"],
    creates=["data/outputs/metrics/cnn_metrics.csv", "data/outputs/metrics/cnn_logits_validation.csv", "data/outputs/metrics/cnn_logits_test.csv"],
)
run_step(
    "branch calibration",
    [sys.executable, "scripts/08_calibrate_branches.py", "--validation-predictions", "data/outputs/metrics/ml_predictions_validation.csv", "--test-predictions", "data/outputs/metrics/ml_predictions_test.csv", "--method", "platt"],
    enabled=RUN_CALIBRATION,
    requires=["data/outputs/metrics/ml_predictions_validation.csv", "data/outputs/metrics/ml_predictions_test.csv"],
    creates=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/calibration_metrics.csv"],
)
run_step(
    "standard fusion",
    [sys.executable, "scripts/09_run_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/fusion_predictions.csv", "data/outputs/metrics/fusion_metrics.csv"],
)
run_step(
    "subgroup and confounding checks",
    [sys.executable, "scripts/10_shift_and_confounding_checks.py", "--predictions", "data/outputs/metrics/fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_SHIFT_CHECKS,
    requires=["data/outputs/metrics/fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/subgroup_metrics.csv"],
)
run_step(
    "basic report assets",
    [sys.executable, "scripts/11_make_report_assets.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/fusion_predictions.csv"],
    enabled=RUN_REPORT_ASSETS,
    requires=["data/processed/metadata_clean.csv"],
    creates=["reports/report_outline.md", "reports/slides_outline.md"],
)

## Stage 6: Publication Extras

These experiments support the publication-grade claims: metadata-only baseline, quality-weighted fusion, abstention, confidence intervals, COUGHVID external validation, and paper tables.

In [ ]:
run_step(
    "metadata-only baseline",
    [sys.executable, "scripts/14_train_metadata_baseline.py", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_METADATA_BASELINE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/metadata_baseline_metrics.csv", "data/outputs/metrics/metadata_predictions_validation.csv", "data/outputs/metrics/metadata_predictions_test.csv"],
)
run_step(
    "quality-weighted fusion",
    [sys.executable, "scripts/16_run_quality_weighted_fusion.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--quality", "data/processed/audio_quality.csv", "--validation-metrics", "data/outputs/metrics/ml_validation_metrics.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_QUALITY_WEIGHTED_FUSION,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv", "data/processed/audio_quality.csv", "data/outputs/metrics/ml_validation_metrics.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/outputs/metrics/quality_weighted_fusion_metrics.csv"],
)
run_step(
    "abstention analysis",
    [sys.executable, "scripts/17_abstention_analysis.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--metadata", "data/processed/metadata_clean.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_ABSTENTION,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv", "data/processed/metadata_clean.csv"],
    creates=["data/outputs/metrics/abstention_decisions.csv", "data/outputs/metrics/abstention_coverage_curve.csv"],
)
run_step(
    "bootstrap CI for quality-weighted fusion",
    [sys.executable, "scripts/15_bootstrap_prediction_metrics.py", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--output", "data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv", "--group-columns", "fusion_method"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_BOOTSTRAP_CI,
    requires=["data/outputs/metrics/quality_weighted_fusion_predictions.csv"],
    creates=["data/outputs/metrics/quality_weighted_fusion_bootstrap_ci.csv"],
)

if COUGHVID_RAW is not None:
    run_step(
        "COUGHVID index",
        [sys.executable, "scripts/13_build_coughvid_index.py", "--raw-dir", COUGHVID_RAW, "--output", "data/interim/coughvid_index.csv", "--min-cough-detected", MIN_COUGH_DETECTED],
        enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_INDEX,
        creates=["data/interim/coughvid_index.csv"],
    )
else:
    print("SKIP COUGHVID index: COUGHVID_RAW is None")

feature_cmd = [sys.executable, "scripts/19_extract_coughvid_features.py", "--index", "data/interim/coughvid_index.csv", "--features-output", "data/processed/coughvid_features_mfcc.csv", "--quality-ok-only"]
if COUGHVID_FEATURE_MAX_ROWS is not None:
    feature_cmd += ["--max-rows", COUGHVID_FEATURE_MAX_ROWS]
run_step(
    "COUGHVID feature extraction",
    feature_cmd,
    enabled=RUN_PUBLICATION_EXTRAS and RUN_COUGHVID_FEATURES,
    requires=["data/interim/coughvid_index.csv"],
    creates=["data/processed/coughvid_features_mfcc.csv"],
    strict_requires=False,
)
run_step(
    "cross-dataset cough evaluation",
    [sys.executable, "scripts/18_cross_dataset_feature_eval.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv", "--modality", "cough", "--model-name", "logistic_regression"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CROSS_DATASET,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["data/outputs/metrics/cross_dataset_predictions.csv", "data/outputs/metrics/cross_dataset_metrics.csv"],
    strict_requires=False,
)
run_step(
    "paper metric tables",
    [sys.executable, "scripts/20_make_paper_tables.py"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAPER_TABLES,
    requires=["data/outputs/metrics/ml_baseline_metrics.csv"],
    creates=["reports/tables/paper_metric_table.csv", "reports/tables/paper_metric_table_raw.csv"],
    strict_requires=False,
)

In [ ]:
import subprocess
import sys
import pandas as pd
import numpy as np

cmds = [
  [
      sys.executable,
      "scripts/23_feature_shift_report.py",
      "--source-features",
      "data/processed/features_mfcc.csv",
      "--external-features",
      "data/processed/coughvid_features_mfcc.csv",
      "--output",
      "reports/tables/feature_shift_report.csv",
      "--summary-output",
      "reports/tables/feature_shift_summary.csv",
  ],
  [
      sys.executable,
      "scripts/24_make_experiment_manifest.py",
  ],
  [
      sys.executable,
      "scripts/12_validate_artifacts.py",
      "--strict",
  ],
]

for cmd in cmds:
  print("\n$", " ".join(cmd))
  result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
  print(result.stdout)
  print(result.stderr)
  result.check_returncode()



In [ ]:
feat = pd.read_csv(PROJECT_ROOT / "data/processed/coughvid_features_mfcc.csv")
cross = pd.read_csv(PROJECT_ROOT / "data/outputs/metrics/cross_dataset_metrics.csv")
shift_summary = pd.read_csv(PROJECT_ROOT / "reports/tables/feature_shift_summary.csv")

print("Full COUGHVID feature shape:", feat.shape)

print("\nFeature labels:")
print(feat["label_binary"].value_counts(dropna=False))

print("\nNaN count:", int(feat.select_dtypes(include="number").isna().sum().sum()))
print("Inf count:", int(np.isinf(feat.select_dtypes(include="number").to_numpy()).sum()))

print("\nCross-dataset metrics:")
display(cross)

print("\nFeature shift summary:")
display(shift_summary)

In [ ]:
import numpy as np
import pandas as pd

idx = pd.read_csv(PROJECT_ROOT / "data/interim/coughvid_index.csv")
feat = pd.read_csv(PROJECT_ROOT / "data/processed/coughvid_features_mfcc.csv")

print("Index shape:", idx.shape)
print("Smoke feature shape:", feat.shape)

print("\nIndex labels:")
print(idx["label_binary"].value_counts(dropna=False))

print("\nSmoke labels:")
print(feat["label_binary"].value_counts(dropna=False))

numeric = feat.select_dtypes(include="number")
print("\nNaN count:", int(numeric.isna().sum().sum()))
print("Inf count:", int(np.isinf(numeric.to_numpy()).sum()))

assert feat.shape[0] == 25, "Smoke extraction did not produce 25 rows"
assert feat.shape[1] == 261, "Unexpected feature column count"
assert numeric.isna().sum().sum() == 0, "NaNs found in features"
assert np.isinf(numeric.to_numpy()).sum() == 0, "Inf values found in features"

print("\nCOUGHVID smoke test is clean.")

In [ ]:
from pathlib import Path
import shutil
import zipfile

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

bundle_dir = PROJECT_ROOT.parent / "results" / "frozen" / "Publication_ExternalValidation_Artifacts_2026-06-12"
zip_path = PROJECT_ROOT.parent / "artifacts" / "bundles" / "Publication_ExternalValidation_Artifacts_2026-06-12.zip"

if bundle_dir.exists():
  shutil.rmtree(bundle_dir)

zip_path.parent.mkdir(parents=True, exist_ok=True)
(bundle_dir / "metrics").mkdir(parents=True)
(bundle_dir / "tables").mkdir(parents=True)
(bundle_dir / "reports").mkdir(parents=True)

files_to_copy = [
  ("data/interim/coughvid_index.csv", "tables/coughvid_index.csv"),
  ("data/processed/coughvid_features_mfcc.csv", "tables/coughvid_features_mfcc.csv"),
  ("data/outputs/metrics/cross_dataset_metrics.csv", "metrics/cross_dataset_metrics.csv"),
  ("data/outputs/metrics/cross_dataset_predictions.csv", "metrics/cross_dataset_predictions.csv"),
  ("reports/tables/feature_shift_report.csv", "tables/feature_shift_report.csv"),
  ("reports/tables/feature_shift_summary.csv", "tables/feature_shift_summary.csv"),
  ("reports/tables/paper_metric_table.csv", "tables/paper_metric_table.csv"),
  ("reports/tables/paper_metric_table_raw.csv", "tables/paper_metric_table_raw.csv"),
  ("reports/experiment_manifest.json", "reports/experiment_manifest.json"),
]

for src_rel, dst_rel in files_to_copy:
  src = PROJECT_ROOT / src_rel
  dst = bundle_dir / dst_rel
  if src.exists():
      shutil.copy2(src, dst)
      print("copied:", src_rel)
  else:
      print("missing:", src_rel)

readme = bundle_dir / "README.md"
readme.write_text(
  """# Publication External Validation Artifacts

Date: 2026-06-12
Scope: COUGHVID v3 external validation after corrected no-leakage Coswara baseline.

Completed:
- COUGHVID v3 extraction
- COUGHVID index
- COUGHVID MFCC feature extraction
- Coswara-to-COUGHVID cough-only external validation
- Feature-shift report
- Paper metric table refresh
- Experiment manifest refresh

Key result:
- Logistic regression cough-only Coswara-to-COUGHVID AUROC: 0.52948
- AUPRC: 0.043007
- Balanced accuracy: 0.522575
- F1: 0.071313
- External labeled rows: 8331
- High-shift features: 21 / 253

Interpretation:
This is evidence of strong cross-dataset shift. It should be framed as an external stress test, not a failed implementation.
""",
  encoding="utf-8",
)

if zip_path.exists():
  zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
  for file in bundle_dir.rglob("*"):
      z.write(file, file.relative_to(bundle_dir.parent))

print("\nBundle folder:", bundle_dir)
print("Bundle zip:", zip_path)
print("Zip exists:", zip_path.exists())
print("Zip size MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

## Stage 7: Extra Publication Strengthening

These optional diagnostics make the paper stronger: paired model comparison, matched confounding subset, source-vs-external feature shift, and a reproducibility manifest.

In [ ]:
run_step(
    "paired ML model comparison",
    [sys.executable, "scripts/21_paired_model_comparison.py", "--predictions", "data/outputs/metrics/calibrated_branch_predictions.csv", "--baseline-name", "logistic_regression", "--model-column", "model_name", "--group-columns", "modality", "--output", "data/outputs/metrics/paired_model_comparison.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_PAIRED_MODEL_COMPARISON,
    requires=["data/outputs/metrics/calibrated_branch_predictions.csv"],
    creates=["data/outputs/metrics/paired_model_comparison.csv"],
    strict_requires=False,
)
run_step(
    "confounding matched subset",
    [sys.executable, "scripts/22_confounding_matching.py", "--metadata", "data/processed/metadata_clean.csv", "--predictions", "data/outputs/metrics/quality_weighted_fusion_predictions.csv", "--covariates", "age_bucket", "gender"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_CONFOUNDING_MATCHING,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/metadata_matched.csv", "reports/tables/confounding_balance.csv"],
    strict_requires=False,
)
run_step(
    "feature shift report",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", "data/processed/features_mfcc.csv", "--external-features", "data/processed/coughvid_features_mfcc.csv"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_FEATURE_SHIFT_REPORT,
    requires=["data/processed/features_mfcc.csv", "data/processed/coughvid_features_mfcc.csv"],
    creates=["reports/tables/feature_shift_report.csv", "reports/tables/feature_shift_summary.csv"],
    strict_requires=False,
)
run_step(
    "experiment manifest",
    [sys.executable, "scripts/24_make_experiment_manifest.py", "--run-name", "covid_audio_publication_run"],
    enabled=RUN_PUBLICATION_EXTRAS and RUN_EXPERIMENT_MANIFEST,
    creates=["reports/experiment_manifest.json"],
)

## Final Dashboard

Use this at the end of the run to see exactly what was produced.

In [ ]:
import pandas as pd

final_dashboard = pd.DataFrame(
    [{"path": path, "exists": path_exists(path), "size_bytes": p(path).stat().st_size if p(path).exists() else 0} for path in CORE_ARTIFACTS]
)
display(final_dashboard)

for path in [
    "data/outputs/metrics/ml_baseline_metrics.csv",
    "data/outputs/metrics/fusion_metrics.csv",
    "data/outputs/metrics/quality_weighted_fusion_metrics.csv",
    "reports/tables/paper_metric_table.csv",
    "data/outputs/metrics/paired_model_comparison.csv",
    "reports/tables/confounding_balance.csv",
    "reports/tables/feature_shift_report.csv",
    "reports/experiment_manifest.json",
]:
    if path_exists(path):
        print()
        print(path)
        display(pd.read_csv(p(path)).head(20))

print("Run finished")

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

def run_cmd(name, args):
  print(f"\n## {name}")
  print("$", " ".join(str(a) for a in args))
  subprocess.run([str(a) for a in args], check=True)

required_files = [
  "data/processed/features_mfcc.csv",
  "data/processed/coughvid_features_mfcc.csv",
  "reports/tables/feature_shift_report.csv",
  "data/outputs/metrics/ml_validation_metrics.csv",
]

missing = [p for p in required_files if not Path(p).exists()]
if missing:
  raise FileNotFoundError(f"Missing required files. Stop here: {missing}")

run_cmd("install optional rescue dependencies", [sys.executable, "-m", "pip", "install", "-r", "requirements-optional.txt"])
run_cmd("environment check", [sys.executable, "scripts/00_check_environment.py"])

print("\nReady for rescue experiments.")

In [ ]:
run_cmd("focused rescue tests", [sys.executable, "-m", "pytest", "tests/test_rescue_experiments.py", "-q"])
run_cmd("full test suite", [sys.executable, "-m", "pytest", "tests", "-q"])

In [ ]:
run_cmd(
  "COUGHVID internal baseline smoke",
  [
      sys.executable,
      "scripts/26_run_coughvid_internal_baseline.py",
      "--features", "data/processed/coughvid_features_mfcc.csv",
      "--models", "logistic_regression", "random_forest",
      "--split-features-output", "data/processed/coughvid_features_mfcc_internal_split_smoke.csv",
      "--predictions-output", "data/outputs/metrics/coughvid_internal_predictions_smoke.csv",
      "--metrics-output", "data/outputs/metrics/coughvid_internal_metrics_smoke.csv",
      "--bootstrap-output", "data/outputs/metrics/coughvid_internal_bootstrap_ci_smoke.csv",
      "--n-bootstraps", "50",
  ],
)

run_cmd(
  "external model grid smoke",
  [
      sys.executable,
      "scripts/25_run_external_model_grid.py",
      "--source-features", "data/processed/features_mfcc.csv",
      "--external-features", "data/processed/coughvid_features_mfcc.csv",
      "--feature-shift-report", "reports/tables/feature_shift_report.csv",
      "--models", "logistic_regression", "random_forest",
      "--feature-strategies", "all", "drop_high_shift", "top_stable_50",
      "--predictions-output", "data/outputs/metrics/external_model_grid_predictions_smoke.csv",
      "--metrics-output", "data/outputs/metrics/external_model_grid_metrics_smoke.csv",
      "--bootstrap-output", "data/outputs/metrics/external_model_grid_bootstrap_ci_smoke.csv",
      "--n-bootstraps", "50",
  ],
)

In [ ]:
run_cmd(
  "COUGHVID internal baseline final",
  [
      sys.executable,
      "scripts/26_run_coughvid_internal_baseline.py",
      "--features", "data/processed/coughvid_features_mfcc.csv",
      "--n-bootstraps", "1000",
  ],
)

run_cmd(
  "external model grid final",
  [
      sys.executable,
      "scripts/25_run_external_model_grid.py",
      "--source-features", "data/processed/features_mfcc.csv",
      "--external-features", "data/processed/coughvid_features_mfcc.csv",
      "--feature-shift-report", "reports/tables/feature_shift_report.csv",
      "--n-bootstraps", "1000",
  ],
)

run_cmd("paper metric tables", [sys.executable, "scripts/20_make_paper_tables.py"])
run_cmd("experiment manifest", [sys.executable, "scripts/24_make_experiment_manifest.py"])
run_cmd("artifact validation", [sys.executable, "scripts/12_validate_artifacts.py", "--strict"])

In [ ]:
outputs = [
  "data/outputs/metrics/coughvid_internal_metrics.csv",
  "data/outputs/metrics/coughvid_internal_bootstrap_ci.csv",
  "data/outputs/metrics/external_model_grid_metrics.csv",
  "data/outputs/metrics/external_model_grid_bootstrap_ci.csv",
  "reports/tables/paper_metric_table.csv",
  "reports/experiment_manifest.json",
]

for p in outputs:
  path = Path(p)
  print(p, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)

print("\nCOUGHVID internal baseline:")
display(pd.read_csv("data/outputs/metrics/coughvid_internal_metrics.csv").sort_values(["auroc", "auprc"], ascending=False))

print("\nExternal Coswara-to-COUGHVID model grid:")
display(pd.read_csv("data/outputs/metrics/external_model_grid_metrics.csv").sort_values(["auroc", "auprc"], ascending=False))

print("\nPaper table tail:")
display(pd.read_csv("reports/tables/paper_metric_table.csv").tail(40))

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent

files = [
  "scripts/25_run_external_model_grid.py",
  "scripts/26_run_coughvid_internal_baseline.py",
  "src/covid_audio_btp/rescue_experiments.py",
  "tests/test_rescue_experiments.py",
]

for f in files:
  p = PROJECT_ROOT / f
  print(f, p.exists())

In [ ]:
import os
import sys
import subprocess

os.chdir(PROJECT_ROOT)

def run_cmd(name, args):
  print(f"\n## {name}")
  print("$", " ".join(str(a) for a in args))
  subprocess.run([str(a) for a in args], check=True)

run_cmd("paper metric tables", [sys.executable, "scripts/20_make_paper_tables.py"])
run_cmd("experiment manifest", [sys.executable, "scripts/24_make_experiment_manifest.py"])
run_cmd("artifact validation", [sys.executable, "scripts/12_validate_artifacts.py", "--strict"])

In [ ]:
import pandas as pd

paper = pd.read_csv("reports/tables/paper_metric_table.csv")

print(paper.columns.tolist())

display(
  paper[paper["table_source"].eq("external_model_grid_metrics")]
  [["table_source", "model_name", "modality", "feature_strategy", "calibration_method", "n_samples", "auroc", "auprc", "balanced_accuracy", "f1"]]
  .sort_values(["auroc", "auprc"], ascending=False)
)

In [ ]:
print(SSL_CHECKPOINT_PATH, SSL_CHECKPOINT_PATH.exists())
print(SSL_SOURCE_DIR, SSL_SOURCE_DIR.exists())
print((SSL_SOURCE_DIR / "BEATs.py").exists())

## Stage 8: Controlled Representation Experiments

This section is disabled by default. Flip only the representation stage you want to run. The intended order is OpenSMILE smoke, OpenSMILE full cough external/internal checks, then one SSL backend at a time.


In [17]:
# Representation experiment controls. Defaults are safe for Run All.
RUN_OPENSMILE_SMOKE = False
RUN_OPENSMILE_COSWARA_COUGH = False
RUN_OPENSMILE_COUGHVID_COUGH = False
RUN_OPENSMILE_SHIFT = False
RUN_OPENSMILE_EXTERNAL_GRID = False
RUN_OPENSMILE_COUGHVID_INTERNAL = False

# SSL_BACKEND choices: wav2vec2, beats, panns
SSL_BACKEND = "panns"
SSL_CHECKPOINT_PATH = PROJECT_ROOT / "models/external/panns/Cnn14_mAP_0.431.pth"
SSL_SOURCE_DIR = PROJECT_ROOT / "external/audioset_tagging_cnn"

RUN_SSL_SMOKE = False
RUN_SSL_COSWARA_COUGH = False
RUN_SSL_COUGHVID_COUGH = False
RUN_SSL_SHIFT = False
RUN_SSL_EXTERNAL_GRID = True
RUN_SSL_COUGHVID_INTERNAL = True

REPRESENTATION_QUALITY_MODE = "quality_ok_only"
REPRESENTATION_MODALITY = "cough"
REPRESENTATION_SMOKE_ROWS = 25
REPRESENTATION_BATCH_SIZE = 8
REPRESENTATION_BOOTSTRAPS = 1000

OPENSMILE_SOURCE_FEATURES = "data/processed/features_opensmile_egemaps_coswara_cough.csv"
OPENSMILE_EXTERNAL_FEATURES = "data/processed/features_opensmile_egemaps_coughvid_cough.csv"
OPENSMILE_SHIFT_REPORT = "reports/tables/feature_shift_opensmile_egemaps_cough.csv"
OPENSMILE_SHIFT_SUMMARY = "reports/tables/feature_shift_opensmile_egemaps_summary.csv"

ssl_slug = SSL_BACKEND.lower().replace("-", "_")
SSL_SOURCE_FEATURES = f"data/processed/features_{ssl_slug}_coswara_cough.csv"
SSL_EXTERNAL_FEATURES = f"data/processed/features_{ssl_slug}_coughvid_cough.csv"
SSL_SHIFT_REPORT = f"reports/tables/feature_shift_{ssl_slug}_cough.csv"
SSL_SHIFT_SUMMARY = f"reports/tables/feature_shift_{ssl_slug}_summary.csv"

print("Representation controls ready")
print("SSL backend:", SSL_BACKEND)
print("SSL checkpoint:", SSL_CHECKPOINT_PATH)
print("SSL source dir:", SSL_SOURCE_DIR)

Representation controls ready
SSL backend: panns
SSL checkpoint: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/models/external/panns/Cnn14_mAP_0.431.pth
SSL source dir: /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/external/audioset_tagging_cnn


In [18]:
# OpenSMILE eGeMAPSv02: stronger handcrafted baseline.
run_step(
    "OpenSMILE Coswara cough smoke",
    [sys.executable, "scripts/27_extract_opensmile_features.py", "--metadata", "data/processed/metadata_clean.csv", "--output", "data/processed/features_opensmile_egemaps_coswara_cough_smoke.csv", "--feature-set", "egemaps", "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY, "--max-rows", REPRESENTATION_SMOKE_ROWS],
    enabled=RUN_OPENSMILE_SMOKE,
    requires=["data/processed/metadata_clean.csv"],
    creates=["data/processed/features_opensmile_egemaps_coswara_cough_smoke.csv"],
    strict_requires=False,
)
run_step(
    "OpenSMILE Coswara cough full",
    [sys.executable, "scripts/27_extract_opensmile_features.py", "--metadata", "data/processed/metadata_clean.csv", "--output", OPENSMILE_SOURCE_FEATURES, "--feature-set", "egemaps", "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY],
    enabled=RUN_OPENSMILE_COSWARA_COUGH,
    requires=["data/processed/metadata_clean.csv"],
    creates=[OPENSMILE_SOURCE_FEATURES],
    strict_requires=False,
)
run_step(
    "OpenSMILE COUGHVID cough full",
    [sys.executable, "scripts/27_extract_opensmile_features.py", "--metadata", "data/interim/coughvid_index.csv", "--output", OPENSMILE_EXTERNAL_FEATURES, "--feature-set", "egemaps", "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY, "--split-name", "external"],
    enabled=RUN_OPENSMILE_COUGHVID_COUGH,
    requires=["data/interim/coughvid_index.csv"],
    creates=[OPENSMILE_EXTERNAL_FEATURES],
    strict_requires=False,
)
run_step(
    "OpenSMILE feature shift",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", OPENSMILE_SOURCE_FEATURES, "--external-features", OPENSMILE_EXTERNAL_FEATURES, "--output", OPENSMILE_SHIFT_REPORT, "--summary-output", OPENSMILE_SHIFT_SUMMARY],
    enabled=RUN_OPENSMILE_SHIFT,
    requires=[OPENSMILE_SOURCE_FEATURES, OPENSMILE_EXTERNAL_FEATURES],
    creates=[OPENSMILE_SHIFT_REPORT, OPENSMILE_SHIFT_SUMMARY],
    strict_requires=False,
)
run_step(
    "OpenSMILE external model grid",
    [sys.executable, "scripts/25_run_external_model_grid.py", "--source-features", OPENSMILE_SOURCE_FEATURES, "--external-features", OPENSMILE_EXTERNAL_FEATURES, "--feature-shift-report", OPENSMILE_SHIFT_REPORT, "--predictions-output", "data/outputs/metrics/external_model_grid_opensmile_egemaps_predictions.csv", "--metrics-output", "data/outputs/metrics/external_model_grid_opensmile_egemaps_metrics.csv", "--bootstrap-output", "data/outputs/metrics/external_model_grid_opensmile_egemaps_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_OPENSMILE_EXTERNAL_GRID,
    requires=[OPENSMILE_SOURCE_FEATURES, OPENSMILE_EXTERNAL_FEATURES, OPENSMILE_SHIFT_REPORT],
    creates=["data/outputs/metrics/external_model_grid_opensmile_egemaps_metrics.csv"],
    strict_requires=False,
)
run_step(
    "OpenSMILE COUGHVID internal baseline",
    [sys.executable, "scripts/26_run_coughvid_internal_baseline.py", "--features", OPENSMILE_EXTERNAL_FEATURES, "--split-features-output", "data/processed/features_opensmile_egemaps_coughvid_internal_split.csv", "--predictions-output", "data/outputs/metrics/coughvid_internal_opensmile_egemaps_predictions.csv", "--metrics-output", "data/outputs/metrics/coughvid_internal_opensmile_egemaps_metrics.csv", "--bootstrap-output", "data/outputs/metrics/coughvid_internal_opensmile_egemaps_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_OPENSMILE_COUGHVID_INTERNAL,
    requires=[OPENSMILE_EXTERNAL_FEATURES],
    creates=["data/outputs/metrics/coughvid_internal_opensmile_egemaps_metrics.csv"],
    strict_requires=False,
)


SKIP OpenSMILE Coswara cough smoke: disabled
SKIP OpenSMILE Coswara cough full: disabled
SKIP OpenSMILE COUGHVID cough full: disabled
SKIP OpenSMILE feature shift: disabled
SKIP OpenSMILE external model grid: disabled
SKIP OpenSMILE COUGHVID internal baseline: disabled


False

In [19]:
# SSL embeddings: run exactly one backend at a time.
def ssl_embedding_cmd(metadata, output, *, split_name=None, max_rows=None):
    cmd = [sys.executable, "scripts/28_extract_ssl_embeddings.py", "--metadata", metadata, "--output", output, "--backend", SSL_BACKEND, "--quality-mode", REPRESENTATION_QUALITY_MODE, "--modality", REPRESENTATION_MODALITY, "--batch-size", REPRESENTATION_BATCH_SIZE]
    if SSL_BACKEND.lower().replace("-", "_") in {"beats", "beats_official", "panns", "panns_cnn14", "panns_cnn14_official"}:
        cmd += ["--checkpoint-path", SSL_CHECKPOINT_PATH, "--source-dir", SSL_SOURCE_DIR]
    if split_name is not None:
        cmd += ["--split-name", split_name]
    if max_rows is not None:
        cmd += ["--max-rows", max_rows]
    return cmd

ssl_model_requires = []
if SSL_BACKEND.lower().replace("-", "_") in {"beats", "beats_official", "panns", "panns_cnn14", "panns_cnn14_official"}:
    ssl_model_requires = [SSL_CHECKPOINT_PATH, SSL_SOURCE_DIR]

run_step(
    f"{SSL_BACKEND} Coswara cough smoke",
    ssl_embedding_cmd("data/processed/metadata_clean.csv", f"data/processed/features_{ssl_slug}_coswara_cough_smoke.csv", max_rows=REPRESENTATION_SMOKE_ROWS),
    enabled=RUN_SSL_SMOKE,
    requires=["data/processed/metadata_clean.csv", *ssl_model_requires],
    creates=[f"data/processed/features_{ssl_slug}_coswara_cough_smoke.csv"],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} Coswara cough full",
    ssl_embedding_cmd("data/processed/metadata_clean.csv", SSL_SOURCE_FEATURES),
    enabled=RUN_SSL_COSWARA_COUGH,
    requires=["data/processed/metadata_clean.csv", *ssl_model_requires],
    creates=[SSL_SOURCE_FEATURES],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} COUGHVID cough full",
    ssl_embedding_cmd("data/interim/coughvid_index.csv", SSL_EXTERNAL_FEATURES, split_name="external"),
    enabled=RUN_SSL_COUGHVID_COUGH,
    requires=["data/interim/coughvid_index.csv", *ssl_model_requires],
    creates=[SSL_EXTERNAL_FEATURES],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} feature shift",
    [sys.executable, "scripts/23_feature_shift_report.py", "--source-features", SSL_SOURCE_FEATURES, "--external-features", SSL_EXTERNAL_FEATURES, "--output", SSL_SHIFT_REPORT, "--summary-output", SSL_SHIFT_SUMMARY],
    enabled=RUN_SSL_SHIFT,
    requires=[SSL_SOURCE_FEATURES, SSL_EXTERNAL_FEATURES],
    creates=[SSL_SHIFT_REPORT, SSL_SHIFT_SUMMARY],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} external model grid",
    [sys.executable, "scripts/25_run_external_model_grid.py", "--source-features", SSL_SOURCE_FEATURES, "--external-features", SSL_EXTERNAL_FEATURES, "--feature-shift-report", SSL_SHIFT_REPORT, "--predictions-output", f"data/outputs/metrics/external_model_grid_{ssl_slug}_predictions.csv", "--metrics-output", f"data/outputs/metrics/external_model_grid_{ssl_slug}_metrics.csv", "--bootstrap-output", f"data/outputs/metrics/external_model_grid_{ssl_slug}_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_SSL_EXTERNAL_GRID,
    requires=[SSL_SOURCE_FEATURES, SSL_EXTERNAL_FEATURES, SSL_SHIFT_REPORT],
    creates=[f"data/outputs/metrics/external_model_grid_{ssl_slug}_metrics.csv"],
    strict_requires=False,
)
run_step(
    f"{SSL_BACKEND} COUGHVID internal baseline",
    [sys.executable, "scripts/26_run_coughvid_internal_baseline.py", "--features", SSL_EXTERNAL_FEATURES, "--split-features-output", f"data/processed/features_{ssl_slug}_coughvid_internal_split.csv", "--predictions-output", f"data/outputs/metrics/coughvid_internal_{ssl_slug}_predictions.csv", "--metrics-output", f"data/outputs/metrics/coughvid_internal_{ssl_slug}_metrics.csv", "--bootstrap-output", f"data/outputs/metrics/coughvid_internal_{ssl_slug}_bootstrap_ci.csv", "--n-bootstraps", REPRESENTATION_BOOTSTRAPS],
    enabled=RUN_SSL_COUGHVID_INTERNAL,
    requires=[SSL_EXTERNAL_FEATURES],
    creates=[f"data/outputs/metrics/coughvid_internal_{ssl_slug}_metrics.csv"],
    strict_requires=False,
)


SKIP panns Coswara cough smoke: disabled
SKIP panns Coswara cough full: disabled
SKIP panns COUGHVID cough full: disabled
SKIP panns feature shift: disabled

## panns external model grid
$ /home/covid/Desktop/Covid-19-BTP/covid_audio_btp/.venv/bin/python scripts/25_run_external_model_grid.py --source-features data/processed/features_panns_coswara_cough.csv --external-features data/processed/features_panns_coughvid_cough.csv --feature-shift-report reports/tables/feature_shift_panns_cough.csv --predictions-output data/outputs/metrics/external_model_grid_panns_predictions.csv --metrics-output data/outputs/metrics/external_model_grid_panns_metrics.csv --bootstrap-output data/outputs/metrics/external_model_grid_panns_bootstrap_ci.csv --n-bootstraps 1000
Ran logistic_regression/all: AUROC=0.4988, AUPRC=0.0347, features=1452
Ran logistic_regression/drop_high_shift: AUROC=0.5024, AUPRC=0.0349, features=1450
SKIP logistic_regression/top_stable_50: Feature strategy top_stable_50 selected no trai

True

In [ ]:
import pandas as pd
import numpy as np

p = PROJECT_ROOT / "data/processed/features_opensmile_egemaps_coswara_cough_smoke.csv"
df = pd.read_csv(p)

print("shape:", df.shape)
print("representation:", df["representation"].value_counts(dropna=False))
print("labels:", df["label_binary"].value_counts(dropna=False))
print("splits:", df["split"].value_counts(dropna=False))

num = df.select_dtypes(include="number")
print("NaN:", int(num.isna().sum().sum()))
print("Inf:", int(np.isinf(num.to_numpy()).sum()))
display(df.head())

In [ ]:
from pathlib import Path

files = [
  "data/processed/features_opensmile_egemaps_coswara_cough.csv",
  "data/processed/features_opensmile_egemaps_coughvid_cough.csv",
  "reports/tables/feature_shift_opensmile_egemaps_cough.csv",
  "reports/tables/feature_shift_opensmile_egemaps_summary.csv",
  "data/outputs/metrics/external_model_grid_opensmile_egemaps_metrics.csv",
  "data/outputs/metrics/coughvid_internal_opensmile_egemaps_metrics.csv",
]

for f in files:
  path = PROJECT_ROOT / f
  print(f, path.exists(), path.stat().st_size if path.exists() else 0)

In [ ]:
import pandas as pd
import numpy as np

beats_smoke_path = PROJECT_ROOT / "data/processed/features_beats_coswara_cough_smoke.csv"
df = pd.read_csv(beats_smoke_path)

print("shape:", df.shape)
print("representation:", df["representation"].value_counts(dropna=False))
print("labels:", df["label_binary"].value_counts(dropna=False))
print("splits:", df["split"].value_counts(dropna=False))
print("NaN:", int(num.isna().sum().sum()))
print("Inf:", int(np.isinf(num.to_numpy()).sum()))
display(df.head())

In [ ]:
run_step(
  "paper metric tables",
  [sys.executable, "scripts/20_make_paper_tables.py"],
  enabled=True,
  creates=["reports/tables/paper_metric_table.csv", "reports/tables/paper_metric_table_raw.csv"],
  force=True,
)

run_step(
  "experiment manifest",
  [sys.executable, "scripts/24_make_experiment_manifest.py"],
  enabled=True,
  creates=["reports/experiment_manifest.json"],
  force=True,
)

run_step(
  "artifact validation",
  [sys.executable, "scripts/12_validate_artifacts.py", "--strict"],
  enabled=True,
  creates=[],
  force=True,
)

In [ ]:
import pandas as pd

display(
  pd.read_csv(PROJECT_ROOT / "data/outputs/metrics/external_model_grid_beats_metrics.csv")
  .sort_values(["auroc", "auprc"], ascending=False)
)

display(
  pd.read_csv(PROJECT_ROOT / "data/outputs/metrics/coughvid_internal_beats_metrics.csv")
  .sort_values(["auroc", "auprc"], ascending=False)
)

display(pd.read_csv(PROJECT_ROOT / "reports/tables/paper_metric_table.csv").tail(100))